# VisiumHD sp-SVC Reconstruction Impact

A route-specific, reproducible analysis of cluster representation change and spatial impact.

## 1. Question, route semantics, and endpoint hierarchy

Raw is the original spatial expression representation. Reconstructed expression is the final sp-SVC representation; the only comparison is Raw Leiden to reconstructed-expression Leiden.

| Endpoint | Primary metrics | Supporting evidence |
|---|---|---|
| Cluster representation change | ST-unit change; balanced cluster change | ARI; normalized contingency; resolution or identity audit |
| Spatial impact and high-diversity Region | Window change; Raw/Recon/Delta Neff; Region area and unit coverage | Level1 anatomy context; support, scale, and threshold checks |

**Evidence boundary.** This notebook describes paired representation and spatial-pattern changes. It does not establish biological mechanism, truth, or clinical relevance.

In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("KMP_WARNINGS", "0")
os.environ.setdefault("OMP_NUM_THREADS", "1")
warnings.filterwarnings("ignore", message="The pynvml package is deprecated")

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from scipy.sparse import SparseEfficiencyWarning

from revise.analysis.reconstruction_impact import (
    compute_spatial_impact,
    file_sha256,
    load_reconstruction_impact_config,
    run_partition_analysis,
    write_analysis_artifacts,
    write_partition_artifacts,
    write_spatial_artifacts,
)

ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "analysis").is_dir():
    raise RuntimeError("Run from the REVISE repository root.")
sns.set_theme(style="white", context="notebook")

MAIN_REGIONS = ["Tumor", "Normal", "Interface"]
SUMMARY_REGIONS = ["Overall", *MAIN_REGIONS]
ANATOMY_COLORS = {
    "Tumor": "#d73027",
    "Normal": "#2c7bb6",
    "Interface": "#fdae61",
    "Other": "#d9d9d9",
}

# VisiumHD analysis-cohort switch: anatomy always keeps the full Level1 context.
USE_FULL_VISIUMHD_COHORT = False
VISIUMHD_SAMPLE_N_UNITS = 30_000

def coordinates(adata):
    values = np.asarray(adata.obsm["spatial"], dtype=float)[:, :2]
    return pd.DataFrame(values, index=adata.obs_names, columns=["x", "y"])

def save_figure(fig, name):
    fig.savefig(OUTPUT_DIR / "figures" / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def window_plot(ax, table, value, title, *, cmap="viridis", vmin=None, vmax=None):
    image = ax.scatter(
        table.window_x,
        table.window_y,
        c=table[value],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=10,
        linewidths=0,
    )
    ax.set(title=title, xlabel="x (um)", ylabel="y (um)", aspect="equal")
    ax.invert_yaxis()
    plt.colorbar(image, ax=ax, shrink=0.72)

def anatomy_map(ax, anatomy, title, *, focus=None):
    background = [ANATOMY_COLORS["Other"]] * len(anatomy)
    ax.scatter(anatomy.window_x, anatomy.window_y, c=background, s=9, linewidths=0)
    if focus is None:
        colors = anatomy.level1_region.map(ANATOMY_COLORS)
        ax.scatter(anatomy.window_x, anatomy.window_y, c=colors, s=9, linewidths=0)
        handles = [
            Line2D([0], [0], marker="o", linestyle="", color=ANATOMY_COLORS[region], label=region)
            for region in [*MAIN_REGIONS, "Other"]
        ]
        ax.legend(handles=handles, title="Anatomy context", loc="upper right")
    else:
        selected = anatomy.loc[anatomy.level1_region == focus]
        ax.scatter(
            selected.window_x,
            selected.window_y,
            c=ANATOMY_COLORS[focus],
            s=10,
            linewidths=0,
        )
    ax.set(title=title, xlabel="x (um)", ylabel="y (um)", aspect="equal")
    ax.invert_yaxis()

def iqr_text(row, metric):
    return f"{row[f'median_{metric}']:.3f} [{row[f'q1_{metric}']:.3f}, {row[f'q3_{metric}']:.3f}]"

def selected_regions(table):
    present = [region for region in SUMMARY_REGIONS if region in set(table.level1_region)]
    return table.set_index("level1_region").loc[present].reset_index()

def matched_cluster_colors(comparison):
    raw_labels = sorted(comparison.contingency.index.astype(str))
    base = list(plt.get_cmap("tab20").colors)
    raw_colors = {label: base[i % len(base)] for i, label in enumerate(raw_labels)}
    recon_colors = {}
    matched = comparison.mapping.loc[comparison.mapping.matched & comparison.mapping.raw_cluster.notna()]
    for row in matched.itertuples():
        recon_colors[str(row.recon_cluster)] = raw_colors[str(row.raw_cluster)]
    unmatched = sorted(set(comparison.contingency.columns.astype(str)) - set(recon_colors))
    novel = list(plt.get_cmap("Set2").colors)
    recon_colors.update({label: novel[i % len(novel)] for i, label in enumerate(unmatched)})
    return raw_colors, recon_colors

## 2. Input and spatial-scale audit

Observation IDs are paired strictly. Coordinates are converted to microns before assigning non-overlapping windows. The primary physical scale is 8 um per cell-equivalent and 5x = 40 um per window.

In [ ]:
CONFIG = load_reconstruction_impact_config(
    ROOT / "configs" / "analysis" / "reconstruction_impact_visiumhd_p1crc.yaml"
)
OUTPUT_DIR = Path(os.environ.get("REVISE_ANALYSIS_OUTPUT_ROOT", CONFIG["output"]["dir"]))
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = ROOT / OUTPUT_DIR
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

comparison_config = CONFIG["partition_change"]["comparisons"][0]
RAW_PATH = ROOT / comparison_config["raw_h5ad"]
RECON_PATH = ROOT / comparison_config["reconstructed_spatial_h5ad"]
raw_context = ad.read_h5ad(RAW_PATH, backed="r")
reconstructed_context = ad.read_h5ad(RECON_PATH, backed="r")
reconstructed_ids = reconstructed_context.obs_names.copy()
if not set(reconstructed_ids) <= set(raw_context.obs_names):
    raise ValueError("Reconstructed IDs must occur in Raw.")

level1_col = comparison_config["level1_column"]
spatial = CONFIG["spatial_region"]
raw_full_coords = coordinates(raw_context)
raw_full_level1 = raw_context.obs[level1_col].astype(str).copy()
raw_full_n_units, raw_full_n_genes = raw_context.n_obs, raw_context.n_vars
reconstructed_full_n_genes = reconstructed_context.n_vars
sample_n = None if USE_FULL_VISIUMHD_COHORT else VISIUMHD_SAMPLE_N_UNITS
if not sample_n:
    raw_paired = raw_context[reconstructed_ids].to_memory()
    reconstructed = reconstructed_context.to_memory()
    raw_context.file.close()
    reconstructed_context.file.close()

input_rows = [
    {
        "Role": "Full Raw Level1 context",
        "Units": raw_full_n_units,
        "Genes": raw_full_n_genes,
        "Pairing": "context",
    },
    {
        "Role": "Strict paired reconstruction carrier",
        "Units": len(reconstructed_ids),
        "Genes": reconstructed_full_n_genes,
        "Pairing": "strict same-ID",
    },
]
if sample_n:
    input_rows.append({
        "Role": "Analysis paired cohort",
        "Units": int(sample_n),
        "Genes": reconstructed_full_n_genes,
        "Pairing": "deterministic same-ID sample",
    })
input_overview = pd.DataFrame(input_rows)
scale_overview = pd.DataFrame([{
    "Microns per coordinate": spatial["microns_per_coordinate"],
    "Cell-equivalent (um)": spatial["cell_equivalent_um"],
    "Primary window (um)": spatial["cell_equivalent_um"] * spatial["main_window_multiplier"],
}])
display(input_overview)
display(scale_overview)

## 3. Cluster representation change

| Role | Metric | Interpretation |
|---|---|---|
| Primary | ST-unit change | Fraction of paired units changing cluster after global Hungarian alignment |
| Primary | Balanced cluster change | One minus matched macro-F1; clusters contribute equally |
| Diagnostic | ARI | Label-permutation-invariant partition agreement |
| Structural evidence | Normalized contingency | Raw-cluster retention, splitting, and mixing |

In [ ]:
seed = int(CONFIG["partition_change"]["random_state"])
if sample_n:
    analysis_ids = pd.Index(
        np.random.default_rng(seed).choice(
            reconstructed_ids.to_numpy(), size=int(sample_n), replace=False
        )
    ).sort_values()
    raw_partition = raw_context[analysis_ids].to_memory()
    recon_partition = reconstructed_context[analysis_ids].to_memory()
    sampling_mode = f"deterministic_same_id_sample_{sample_n}"
else:
    analysis_ids = reconstructed_ids
    raw_partition, recon_partition = raw_paired, reconstructed
    sampling_mode = "full_paired_cohort"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)
    warnings.filterwarnings("ignore", message="Some cells have zero counts")
    partition = run_partition_analysis(
        raw_partition,
        recon_partition,
        level1_col=level1_col,
        final_cluster_key=comparison_config.get("reconstructed_cluster_key"),
        route_kind=CONFIG["route_kind"],
        resolution_mode=CONFIG["partition_change"]["mode"],
        resolution_candidates=CONFIG["partition_change"]["level1_resolution_candidates"],
        within_level1_resolution=CONFIG["partition_change"]["within_level1_resolution"],
        random_state=seed,
        n_top_genes=CONFIG["partition_change"]["n_top_genes"],
    )
edge_name, comparison = next(iter(partition.comparisons.items()))

if sample_n:
    raw_paired = raw_partition
    reconstructed = recon_partition
    raw_context.file.close()
    reconstructed_context.file.close()
spatial_partition = partition
spatial_comparison = next(iter(spatial_partition.comparisons.values()))

In [ ]:
normalized = comparison.contingency.div(
    comparison.contingency.sum(axis=1), axis=0
).fillna(0.0)
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(normalized, ax=ax, cmap="mako", vmin=0, vmax=1)
ax.set(
    title="Raw-normalized cluster contingency",
    xlabel="Reconstructed / Final cluster",
    ylabel="Raw Leiden cluster",
)
save_figure(fig, "01_partition_contingency")

In [ ]:
primary = comparison.summary.iloc[0]
primary_table = pd.DataFrame([{
    "Paired units": int(primary.n_units),
    "Raw clusters": int(primary.n_raw_clusters),
    "Recon/Final clusters": int(primary.n_recon_clusters),
    "ST-unit change": primary.st_unit_change_fraction,
    "Balanced cluster change": primary.balanced_cluster_change,
    "ARI": primary.ARI,
}])
display(Markdown(
    f"After global label alignment, **{primary.st_unit_change_fraction:.1%}** of paired units changed cluster. "
    f"Balanced cluster change was **{primary.balanced_cluster_change:.3f}**, with ARI **{primary.ARI:.3f}**."
))
display(primary_table.round(4))

In [ ]:
matched = comparison.mapping.loc[comparison.mapping.raw_cluster.notna()].copy()
largest = matched.nlargest(min(3, len(matched)), "raw_cluster_n")
least_preserved = matched.nsmallest(min(3, len(matched)), "f1")
key_mapping = pd.concat([largest, least_preserved]).drop_duplicates("raw_cluster")
key_mapping = key_mapping.loc[:, [
    "raw_cluster", "recon_cluster", "raw_cluster_n", "overlap_n", "recall", "f1"
]].sort_values("raw_cluster_n", ascending=False)
display(Markdown("**Key mapping rows: largest and least-preserved Raw clusters.**"))
display(key_mapping.round(3))

#### Supporting check: partition definition

In [ ]:
resolution_view = partition.sweep.rename(columns={"ARI": "Level1 ARI"}).copy()
resolution_view["Selected"] = np.isclose(resolution_view["resolution"], partition.resolution)
display(resolution_view[["resolution", "Level1 ARI", "Selected"]].round(4))
display(Markdown(
    f"The Raw Level1 calibration selected resolution **{partition.resolution:g}**; "
    "the same value was used for reconstructed expression."
))

In [ ]:
assignments = spatial_comparison.assignments.set_index("unit_id")
impact = compute_spatial_impact(
    full_coordinates=raw_full_coords,
    full_level1_labels=raw_full_level1,
    paired_coordinates=raw_full_coords.loc[raw_paired.obs_names],
    raw_labels=assignments.raw_cluster,
    reconstructed_labels=assignments.recon_cluster,
    unit_changed=assignments.unit_changed,
    microns_per_coordinate=float(spatial["microns_per_coordinate"]),
    cell_equivalent_um=float(spatial["cell_equivalent_um"]),
    main_window_multiplier=int(spatial["main_window_multiplier"]),
    window_multipliers=list(spatial["window_multipliers"]),
    neff_threshold=float(spatial["neff_threshold"]),
    neff_thresholds=list(spatial["neff_thresholds"]),
    tumor_label=spatial["anatomy_region"]["tumor_label"],
    normal_source_label=spatial["anatomy_region"]["normal_source_label"],
)
input_audit = pd.DataFrame([
    {
        "input": "raw_context",
        "path": str(RAW_PATH),
        "sha256": file_sha256(RAW_PATH),
        "n_units": raw_full_n_units,
        "n_genes": raw_full_n_genes,
    },
    {
        "input": "reconstructed_spatial",
        "path": str(RECON_PATH),
        "sha256": file_sha256(RECON_PATH),
        "n_units": len(reconstructed_ids),
        "n_genes": reconstructed_full_n_genes,
    },
])
manifest = {
    "sample_id": CONFIG["sample"]["id"],
    "route_kind": CONFIG["route_kind"],
    "comparison_edge": edge_name,
    "paired_units": int(raw_paired.n_obs),
    "available_paired_units": int(len(reconstructed_ids)),
    "excluded_paired_units": int(len(reconstructed_ids) - raw_paired.n_obs),
    "anatomy_context_units": int(raw_full_n_units),
    "pairing_status": "strict_same_id",
    "sampling_mode": sampling_mode,
    "random_state": seed,
    "resolution": partition.resolution,
    "resolution_source": partition.resolution_source,
    "resolution_sweep": partition.sweep.to_dict(orient="records"),
    "representation_audit": partition.representation_audit,
}
write_analysis_artifacts(OUTPUT_DIR, config=CONFIG, manifest=manifest, input_audit=input_audit)
write_partition_artifacts(OUTPUT_DIR, partition)
write_spatial_artifacts(OUTPUT_DIR, impact)

## 4. Level1 anatomy context

Anatomy is a fixed spatial context, not a reconstruction endpoint. Tumor-only windows are Tumor, Normal-only (`Intestinal Epithelial`) windows are Normal, and windows containing both are Interface. Candidate and union encodings remain in the saved artifacts.

In [ ]:
anatomy = impact.anatomy_windows
fig, ax = plt.subplots(figsize=(7, 6))
anatomy_map(ax, anatomy, "Level1 anatomy overview")
save_figure(fig, "02_anatomy_overview")

In [ ]:
context_main = impact.anatomy_context_summary.set_index("level1_region").loc[MAIN_REGIONS]
largest_context = context_main.area_fraction.idxmax()
display(Markdown(
    f"Among the three target contexts, **{largest_context}** occupied the largest tissue-window fraction "
    f"({context_main.loc[largest_context, 'area_fraction']:.1%})."
))

### 4A. Tumor

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
anatomy_map(ax, anatomy, "Tumor context", focus="Tumor")
save_figure(fig, "03_anatomy_tumor")

In [ ]:
row = context_main.loc["Tumor"]
display(Markdown(f"Tumor contains **{int(row.full_level1_units):,}** Level1 units across **{int(row.tissue_windows):,}** windows."))

### 4B. Normal

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
anatomy_map(ax, anatomy, "Normal context", focus="Normal")
save_figure(fig, "04_anatomy_normal")

In [ ]:
row = context_main.loc["Normal"]
display(Markdown(f"Normal contains **{int(row.full_level1_units):,}** Level1 units across **{int(row.tissue_windows):,}** windows."))

### 4C. Interface

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
anatomy_map(ax, anatomy, "Tumor-Normal Interface", focus="Interface")
save_figure(fig, "05_anatomy_interface")

In [ ]:
row = context_main.loc["Interface"]
display(Markdown(f"Interface contains **{int(row.full_level1_units):,}** Level1 units across **{int(row.tissue_windows):,}** windows."))
anatomy_table = context_main.reset_index().loc[:, [
    "level1_region", "full_level1_units", "tissue_windows", "area_mm2", "area_fraction"
]]
display(anatomy_table.rename(columns={
    "level1_region": "Anatomy",
    "full_level1_units": "Level1 units",
    "tissue_windows": "Tissue windows",
    "area_mm2": "Area (mm2)",
    "area_fraction": "Area fraction",
}).round(4))

## 5. Spatial impact and high-diversity Region

| Layer | Final metrics | Supporting evidence |
|---|---|---|
| 5A. Cluster-change localization | Unit-change fraction in space and by anatomy | Matched-color Raw/Recon cluster states |
| 5B. Local diversity state and change | Raw, Recon, and Delta Neff; Median [Q1, Q3] | Minimum parent support |
| 5C. High-diversity Region | Region area and unit coverage | Window-scale and Neff-threshold sensitivity |

### 5A. Cluster-change localization

In [ ]:
spatial_frame = impact.unit_assignments.sample(
    min(len(impact.unit_assignments), 50000), random_state=seed
)
raw_colors, recon_colors = matched_cluster_colors(spatial_comparison)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
axes[0].scatter(
    spatial_frame.x,
    spatial_frame.y,
    c=spatial_frame.raw_cluster.map(raw_colors),
    s=1,
    linewidths=0,
)
axes[1].scatter(
    spatial_frame.x,
    spatial_frame.y,
    c=spatial_frame.reconstructed_cluster.map(recon_colors),
    s=1,
    linewidths=0,
)
for ax, title in zip(axes, ["Raw Leiden clusters", "Reconstructed expression clusters"]):
    ax.set(title=title, xlabel="x (um)", ylabel="y (um)", aspect="equal")
axes[0].invert_yaxis()
fig.suptitle("Matched cluster colors are shared across representations")
save_figure(fig, "06_spatial_cluster_states")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
window_plot(
    ax,
    impact.window_metrics,
    "unit_change_fraction",
    "Window ST-unit change fraction",
    cmap="magma",
    vmin=0,
    vmax=1,
)
save_figure(fig, "07_cluster_change_localization")

In [ ]:
change_main = selected_regions(impact.cluster_change_by_anatomy)
change_table = change_main.loc[:, [
    "level1_region", "paired_units", "changed_units", "change_fraction",
    "n_valid_windows", "median_window_change", "q1_window_change", "q3_window_change",
]].copy()
change_table["Window change Median [Q1, Q3]"] = change_table.apply(
    lambda row: iqr_text(row, "window_change"), axis=1
)
top_change = change_table.loc[change_table.level1_region != "Overall"].sort_values(
    "change_fraction", ascending=False
).iloc[0]
display(Markdown(
    f"The highest anatomy-specific changed-unit fraction was in **{top_change.level1_region}** "
    f"({top_change.change_fraction:.1%})."
))
display(change_table.loc[:, [
    "level1_region", "paired_units", "changed_units", "change_fraction",
    "n_valid_windows", "Window change Median [Q1, Q3]",
]].rename(columns={"level1_region": "Context"}).round(4))

### 5B. Local diversity state and change

In [ ]:
valid_metrics = impact.window_metrics.loc[impact.window_metrics.valid_window]
neff_min = float(np.nanmin(valid_metrics[["neff_raw", "neff_recon"]].to_numpy()))
neff_max = float(np.nanmax(valid_metrics[["neff_raw", "neff_recon"]].to_numpy()))
delta_limit = max(float(np.nanmax(np.abs(valid_metrics.delta_neff))), 1e-9)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
window_plot(axes[0], valid_metrics, "neff_raw", "Raw Neff", vmin=neff_min, vmax=neff_max)
window_plot(axes[1], valid_metrics, "neff_recon", "Reconstructed Neff", vmin=neff_min, vmax=neff_max)
window_plot(
    axes[2], valid_metrics, "delta_neff", "Delta Neff",
    cmap="coolwarm", vmin=-delta_limit, vmax=delta_limit,
)
save_figure(fig, "08_local_diversity_state_and_change")

In [ ]:
diversity_main = selected_regions(impact.diversity_by_anatomy)
diversity_table = diversity_main.loc[:, [
    "level1_region", "n_valid_windows",
    "median_neff_raw", "q1_neff_raw", "q3_neff_raw",
    "median_neff_recon", "q1_neff_recon", "q3_neff_recon",
    "median_delta_neff", "q1_delta_neff", "q3_delta_neff",
]].copy()
diversity_table["Raw Neff Median [Q1, Q3]"] = diversity_table.apply(
    lambda row: iqr_text(row, "neff_raw"), axis=1
)
diversity_table["Recon Neff Median [Q1, Q3]"] = diversity_table.apply(
    lambda row: iqr_text(row, "neff_recon"), axis=1
)
diversity_table["Delta Neff Median [Q1, Q3]"] = diversity_table.apply(
    lambda row: iqr_text(row, "delta_neff"), axis=1
)
overall_diversity = diversity_table.set_index("level1_region").loc["Overall"]
anatomy_delta = diversity_table.loc[diversity_table.level1_region != "Overall"]
top_delta = anatomy_delta.sort_values("median_delta_neff", ascending=False).iloc[0]
display(Markdown(
    f"Overall median Neff changed from **{overall_diversity.median_neff_raw:.3f}** to "
    f"**{overall_diversity.median_neff_recon:.3f}**. The largest anatomy-specific median Delta Neff "
    f"was in **{top_delta.level1_region}** ({top_delta.median_delta_neff:.3f})."
))
display(diversity_table.loc[:, [
    "level1_region", "n_valid_windows", "Raw Neff Median [Q1, Q3]",
    "Recon Neff Median [Q1, Q3]", "Delta Neff Median [Q1, Q3]",
]].rename(columns={"level1_region": "Context"}))

#### Supporting check: minimum parent support

In [ ]:
support = impact.support_sensitivity
fig, ax = plt.subplots(figsize=(7, 4.5))
if support.empty:
    ax.text(0.5, 0.5, "Insufficient parent support", ha="center", va="center")
else:
    ax.plot(support.min_parent_units, support.valid_window_fraction, label="valid windows")
    ax.plot(support.min_parent_units, support.retained_unit_fraction, label="retained units")
    ax.axvline(impact.support_selection["min_parent_units"], color="black", linestyle="--")
    ax.legend()
ax.set(
    xlabel="minimum parent units",
    ylabel="fraction",
    title="Support-selection knee",
)
save_figure(fig, "09_parent_support")

In [ ]:
selected_support = impact.support_selection["min_parent_units"]
support_row = support.loc[support.min_parent_units == selected_support].iloc[0]
display(pd.DataFrame([{
    "Selected support": selected_support,
    "P95 occupancy": impact.support_selection["p95_occupancy"],
    "Valid-window fraction": support_row.valid_window_fraction,
    "Retained-unit fraction": support_row.retained_unit_fraction,
}]).round(4))

### 5C. High-diversity Region

**Region-on-anatomy view.** Low-saturation anatomy provides context; saturated windows are valid high-diversity Region windows with reconstructed Neff at least 2.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
background_colors = anatomy.level1_region.map(ANATOMY_COLORS)
ax.scatter(
    anatomy.window_x,
    anatomy.window_y,
    c=background_colors,
    alpha=0.18,
    s=9,
    linewidths=0,
)
region_windows = impact.window_metrics.loc[impact.window_metrics.in_region]
region_colors = region_windows.level1_region.map(ANATOMY_COLORS)
ax.scatter(
    region_windows.window_x,
    region_windows.window_y,
    c=region_colors,
    s=12,
    edgecolors="black",
    linewidths=0.08,
)
ax.set(
    title="High-diversity Region on Level1 anatomy",
    xlabel="x (um)",
    ylabel="y (um)",
    aspect="equal",
)
region_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="", markeredgecolor="black",
        color=ANATOMY_COLORS[region], label=region,
    )
    for region in [*MAIN_REGIONS, "Other"]
]
ax.legend(handles=region_handles, title="Region context", loc="upper right")
ax.invert_yaxis()
save_figure(fig, "10_region_on_anatomy")

In [ ]:
region_main = selected_regions(impact.region_extent_by_anatomy)
overall_region = region_main.set_index("level1_region").loc["Overall"]
anatomy_region = region_main.loc[region_main.level1_region != "Overall"]
top_region = anatomy_region.sort_values("unit_fraction", ascending=False).iloc[0]
display(Markdown(
    f"The Region covered **{overall_region.area_fraction:.1%}** of valid-window area and "
    f"**{overall_region.unit_fraction:.1%}** of valid units. The highest anatomy-specific unit coverage "
    f"was in **{top_region.level1_region}** ({top_region.unit_fraction:.1%})."
))
display(region_main.loc[:, [
    "level1_region", "valid_windows", "region_windows", "region_area_mm2",
    "area_fraction", "valid_units", "region_units", "unit_fraction",
]].rename(columns={
    "level1_region": "Context",
    "region_area_mm2": "Region area (mm2)",
}).round(4))

#### Supporting check: window scale

In [ ]:
scale_check = impact.scale_sensitivity.sort_values("window_multiplier")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(scale_check.window_multiplier, scale_check.region_area_fraction, marker="o", label="area fraction")
ax.plot(scale_check.window_multiplier, scale_check.region_unit_fraction, marker="o", label="unit fraction")
ax.axvline(spatial["main_window_multiplier"], color="black", linestyle="--")
ax.set(xlabel="cell-equivalent multiplier", ylabel="Region fraction", title="Window-scale sensitivity")
ax.legend()
save_figure(fig, "11_region_scale_sensitivity")

In [ ]:
display(scale_check.loc[scale_check.window_multiplier.isin([3, 5, 7]), [
    "window_multiplier", "window_side_length", "region_area_fraction", "region_unit_fraction"
]].round(4))

#### Supporting check: Neff threshold

In [ ]:
threshold_check = impact.threshold_sensitivity.sort_values("threshold")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(threshold_check.threshold, threshold_check.region_area_fraction, marker="o", label="area fraction")
ax.plot(threshold_check.threshold, threshold_check.region_unit_fraction, marker="o", label="unit fraction")
ax.axvline(spatial["neff_threshold"], color="black", linestyle="--")
ax.set(xlabel="Neff threshold", ylabel="Region fraction", title="Region-threshold sensitivity")
ax.legend()
save_figure(fig, "12_region_threshold_sensitivity")

In [ ]:
display(threshold_check.loc[threshold_check.threshold.isin([1.5, 2.0, 2.5]), [
    "threshold", "region_area_fraction", "region_unit_fraction"
]].round(4))

## 6. Take-home results

The final summary reports the two endpoint families only. Supporting metrics above explain calibration and robustness.

In [ ]:
overall_change = change_main.set_index("level1_region").loc["Overall"]
overall_diversity = diversity_main.set_index("level1_region").loc["Overall"]
take_home = pd.DataFrame([
    {
        "Endpoint": "Cluster representation change",
        "Key result": (
            f"ST-unit change {primary.st_unit_change_fraction:.3f}; "
            f"balanced change {primary.balanced_cluster_change:.3f}; ARI {primary.ARI:.3f}"
        ),
    },
    {
        "Endpoint": "Spatial impact and high-diversity Region",
        "Key result": (
            f"median Delta Neff {overall_diversity.median_delta_neff:.3f}; "
            f"Region area {overall_region.area_fraction:.1%}; units {overall_region.unit_fraction:.1%}"
        ),
    },
])
display(take_home)
display(Markdown(
    f"Cluster change was spatially highest in **{top_change.level1_region}**. "
    f"High-diversity Region unit coverage was highest in **{top_region.level1_region}**. "
    "These are descriptive paired results, not biological mechanism or truth validation."
))